# 01 · Baseline — Clasificación de prendas (Caso A, Visión)

**Equipo:** Francisco Arancibia · Benjamín Contreras — Machine Learning (TIEL26)

Transfer learning con **MobileNetV2** preentrenado en ImageNet sobre **Fashion-MNIST** (10 clases). Este notebook reproduce el baseline y sus métricas. Funciona en Google Colab (CPU basta) o en local.

## 1. Dependencias y datos

In [ ]:
!pip -q install torch torchvision onnx onnxruntime scikit-learn pillow psutil matplotlib

In [ ]:
import os, pathlib, subprocess
os.makedirs('data/fashion', exist_ok=True)
base = 'https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/'
for f in ['train-images-idx3-ubyte','train-labels-idx1-ubyte','t10k-images-idx3-ubyte','t10k-labels-idx1-ubyte']:
    p = f'data/fashion/{f}.gz'
    if not os.path.exists(p):
        subprocess.run(['wget','-q','-O',p, base+f+'.gz'], check=True)
sorted(os.listdir('data/fashion'))

## 2. Entrenamiento del baseline
Entrada: 28×28 gris → 96×96 bilineal → 3 canales → normalización ImageNet. Particiones del set de entrenamiento oficial: 12.000 para entrenar y 5.000 para validación (índices en `reports/particion.json`). 2 épocas, AdamW + OneCycle (lr 1e-3), label smoothing 0,1, semilla 42. Sin `--weights-path`, el script descarga los pesos ImageNet oficiales de torchvision.

In [ ]:
!python scripts/train_baseline.py --epochs 2 --train-samples 12000 --img-size 96

## 3. Métricas

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

with open('reports/baseline_metrics.json') as f:
    m = json.load(f)
print('Validación:', m['validacion'], '| Prueba:', m['prueba'])
cm = np.array(m['matriz_confusion'])
clases = ['camiseta','pantalon','sueter','vestido','abrigo','sandalia','camisa','zapatilla','bolso','botin']
plt.figure(figsize=(6,5)); plt.imshow(cm, cmap='Blues'); plt.colorbar()
plt.xticks(range(10), clases, rotation=60); plt.yticks(range(10), clases)
plt.xlabel('Predicción'); plt.ylabel('Real'); plt.title('Matriz de confusión (prueba)'); plt.tight_layout(); plt.show()

## 4. Exportación a ONNX, cuantización INT8, evaluación y benchmark por etapas

In [ ]:
!python scripts/export_onnx.py
!python scripts/evaluate.py
!python scripts/make_test_jpegs.py --n 20
!python scripts/benchmark.py --requests 300

In [ ]:
with open('reports/benchmark.json') as f:
    b = json.load(f)
print(b['hardware'])
for r in b['resultados']:
    print(f"{r['backend']:13s} acc={r['accuracy_test']:.4f} foto p50/p95={r['jpeg_total_p50_ms']}/{r['jpeg_total_p95_ms']} ms "
          f"(decod. {r['jpeg_decodificacion_p50_ms']} + pre {r['preprocesamiento_p50_ms']} + inf {r['inferencia_en_flujo_p50_ms']}) "
          f"| solo inf p50={r['solo_inferencia_p50_ms']} ms | RAM {r['ram_proceso_mb']} MB")